# Stat 220, Unit 2 Homework: A Map of Models

One dataset throughout: `campus_cafe.csv`, 700 days at a campus coffee shop. It is not
one of the datasets from the code companion.

| column | what it is |
|---|---|
| `temp_f` | outside temperature that day |
| `exam_week` | 1 during finals and midterms, 0 otherwise |
| `promo` | 1 if a discount ran that day |
| `foot_traffic` | people who walked past the shop |
| `drinks_sold` | drinks sold that day |
| `revenue` | dollars taken that day |
| `sold_out` | 1 if they ran out of a main item |

**Most of the code is written for you.** Run each cell, read what comes back, and answer the
question. The reasoning is the graded part, not the typing. Start by running this:

```python
import pandas as pd, numpy as np
import statsmodels.api as sm, statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression

cafe = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/campus_cafe.csv")
P = ["drinks_sold", "exam_week", "promo", "temp_f"]
cafe.head()
```

**Problem 1.** *Reading a situation.* No computer. Two or three sentences each.

Part a. The owner wants to know how many drinks to prepare tomorrow, given the forecast temperature. Name the type of `y`, name the model family, and say why.

_Your answer:_



Part b. The owner wants to know whether running a promotion actually raises revenue, because she is deciding whether to keep doing it. Name the type of `y` and the family, and say what makes this a different job from part a.

_Your answer:_



Part c. A classmate suggests a random forest for part b. It would run on that data without complaining. Would you use it, and what would it cost you?

_Your answer:_



**Problem 2.** *The type of `y` picks the model.* Three outcomes in one table.

Part a. Run this. Report the coefficient on `drinks_sold` in a full sentence, with units.

In [ ]:
# a number -> linear regression
m_rev = smf.ols("revenue ~ drinks_sold + exam_week", data=cafe).fit()
print(m_rev.params.round(3))

_Your answer:_



Part b. Run this. Why can this model never predict a negative number of drinks?

In [ ]:
# a count -> Poisson regression
m_cnt = smf.glm("drinks_sold ~ temp_f + exam_week + promo",
                data=cafe, family=sm.families.Poisson()).fit()
print(m_cnt.params.round(4))
print("smallest prediction:", round(m_cnt.predict().min(), 1))

_Your answer:_



Part c. The first model below is the right one for a yes/no outcome. The second forces a linear model onto it. Say what has gone wrong in the second, and what it tells you.

In [ ]:
# a yes/no -> logistic regression
m_out = smf.logit("sold_out ~ drinks_sold", data=cafe).fit(disp=0)
p_ok = m_out.predict()
print(f"logistic fitted probabilities: {p_ok.min():.3f} to {p_ok.max():.3f}")

# the same outcome, forced into a linear model
p_bad = smf.ols("sold_out ~ drinks_sold", data=cafe).fit().predict()
print(f"linear   fitted probabilities: {p_bad.min():.2f} to {p_bad.max():.2f}")
print("how many fall outside 0 to 1:", int(((p_bad < 0) | (p_bad > 1)).sum()))

_Your answer:_



**Problem 3.** *What each kind of model hands back.* Both predict `revenue` from the same four
columns.

Part a. Run this. Report the 95% confidence interval for `exam_week` and say what it means in dollars.

In [ ]:
reg = smf.ols("revenue ~ " + " + ".join(P), data=cafe).fit()
print(reg.summary().tables[1])
print("95% CI for exam_week:", reg.conf_int().loc["exam_week"].round(2).tolist())

_Your answer:_



Part b. Run this. Which predictor does the forest lean on most, and which barely registers?

In [ ]:
forest = RandomForestRegressor(n_estimators=300, random_state=0).fit(cafe[P], cafe["revenue"])
print(pd.Series(forest.feature_importances_, index=P).round(3).to_string())

_Your answer:_



Part c. This checks which quantities each model can produce. For each one the forest cannot, say why not. Use the word distribution.

In [ ]:
for name, model in [("regression", reg), ("forest", forest)]:
    have = [a for a in ["pvalues", "conf_int", "aic"] if hasattr(model, a)]
    print(f"{name:<12} can give you: {have}")

_Your answer:_



**Problem 4.** *Is model A better than model B?* Same rows, same measure, both times.

Part a. Run this. One of the two additions makes AIC go up rather than down. Which, and what does that mean?

In [ ]:
for f in ["revenue ~ drinks_sold",
          "revenue ~ drinks_sold + exam_week",
          "revenue ~ drinks_sold + exam_week + promo"]:
    print(f"{f:<48} AIC = {smf.ols(f, data=cafe).fit().aic:8.1f}")

_Your answer:_



Part b. Run this. Which model would you ship, and does the answer surprise you?

In [ ]:
folds = KFold(5, shuffle=True, random_state=0)   # one fold object, used for both
for name, mod in [("regression", LinearRegression()),
                  ("forest", RandomForestRegressor(n_estimators=300, random_state=0))]:
    mse = -cross_val_score(mod, cafe[P], cafe["revenue"], cv=folds,
                           scoring="neg_mean_squared_error").mean()
    print(f"  {name:<12} cross-validated MSE {mse:7.1f}   RMSE {np.sqrt(mse):5.1f} dollars")

_Your answer:_



Part c. You have AIC values from part a and cross-validated errors from part b. Explain in two sentences why you cannot settle the regression-versus-forest question using AIC.

_Your answer:_



**Problem 5.** *One prediction, with a range on it.*

Part a. Run this. Report the prediction and the interval, and say whether the interval caught the true value.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(cafe[P], cafe["revenue"],
                                      test_size=0.25, random_state=1)
fit = LinearRegression().fit(Xtr, ytr)

pred = fit.predict(Xte.iloc[[0]])[0]           # one held-out day
resid = yte - fit.predict(Xte)                 # errors on days it never saw
lo, hi = np.percentile(resid, [5, 95])

print(f"prediction          : {pred:6.1f}")
print(f"90% interval        : {pred + lo:6.1f} to {pred + hi:.1f}")
print(f"what it actually was: {yte.iloc[0]:6.1f}")

_Your answer:_



Part b. The owner asks for one number and no range. Write the single sentence you would say to her instead, using your numbers from part a.

_Your answer:_



Part c. Look back at your two answers about `exam_week`. The forest gave it an importance near the bottom of the four. The regression gave it a coefficient several dollars away from zero. Explain how both can be true, and say which number you would take to the owner.

_Your answer:_

